# Lab 1. sin, sinf, __sinf on GPU (float and double)
- Array of 10^9 elements: `arr[i] = sin((i % 360) * Pi / 180)`, filled by a GPU kernel.
- Compare 3 functions (sin, sinf, __sinf) for float (4 GB) and double (8 GB): error, kernel time, copy time.
- Reference is calculated on the CPU in double (ground truth).
- GPU: Tesla T4 (Colab). Run cells from top to bottom.

### GPU check
- Shows the GPU model (Tesla T4) and memory (15 GB).

In [ ]:
!nvidia-smi

Mon Oct 12 14:40:53 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   60C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### Part 1. Libraries
- `%%writefile` creates `lab1.cu`, next cells add code with `%%writefile -a`.
- cstdio – printf, cstdlib – malloc/free, cmath – sin/fabs, cuda_runtime – CUDA functions.

In [ ]:
%%writefile lab1.cu
// =====================================================================
// Laboratory Assignment 1
// GPU precision and performance: sin, sinf, __sinf (float and double)
// =====================================================================

// ===== Part 1. Libraries =====
#include <cstdio>
#include <cstdlib>
#include <cmath>
#include <cuda_runtime.h>

Writing lab1.cu


### Part 2. CHECK macro
- Every CUDA function returns an error code.
- If it is not `cudaSuccess`, CHECK prints the error, file and line, and stops the program.

In [ ]:
%%writefile -a lab1.cu

// ===== Part 2. CUDA error checking macro =====
#define CHECK(call)                                                   \
    do {                                                              \
        cudaError_t e = (call);                                       \
        if (e != cudaSuccess) {                                       \
            fprintf(stderr, "CUDA error: %s (%s:%d)\n",               \
                    cudaGetErrorString(e), __FILE__, __LINE__);       \
            exit(1);                                                  \
        }                                                             \
    } while (0)

Appending to lab1.cu


### Part 3. Constants
- N = 10^9 elements, 256 threads per block (multiple of warp size 32), 10 repeats for time averaging.
- Pi is separate for float and double, so the float kernel calculates only in float (double is slow on T4).
- F_SIN / F_SINF / F_INTR = 0 / 1 / 2 – function numbers.

In [ ]:
%%writefile -a lab1.cu

// ===== Part 3. Constants =====
constexpr long long N  = 1000000000LL;   // array size: 10^9
constexpr int THREADS  = 256;            // threads per block
constexpr int REPEATS  = 10;             // kernel repeats for time averaging

constexpr float  PI_F = 3.14159265358979f;         // Pi for float
constexpr double PI_D = 3.14159265358979323846;    // Pi for double

constexpr int F_SIN  = 0;   // sin
constexpr int F_SINF = 1;   // sinf
constexpr int F_INTR = 2;   // __sinf
const char* NAMES[] = {"sin   ", "sinf  ", "__sinf"};

Appending to lab1.cu


### Part 4. Kernels
- `__global__` – runs on the GPU, called from the CPU.
- Index: `i = blockIdx.x * blockDim.x + threadIdx.x`, one thread = one element.
- `if (i >= n) return;` – the last block has extra threads.
- `(i % 360) * Pi / 180` – degrees → radians. `func` selects sin / sinf / __sinf.
- In the double kernel, sinf and __sinf get `(float)x`, because they accept only float.

In [ ]:
%%writefile -a lab1.cu

// ===== Part 4. GPU kernels =====
__global__ void kernel_float(float* arr, long long n, int func) {
    long long i = (long long)blockIdx.x * blockDim.x + threadIdx.x;
    if (i >= n) return;
    float x = (i % 360) * PI_F / 180.0f;
    if      (func == F_SIN)  arr[i] = sin(x);
    else if (func == F_SINF) arr[i] = sinf(x);
    else if (func == F_INTR) arr[i] = __sinf(x);
}

__global__ void kernel_double(double* arr, long long n, int func) {
    long long i = (long long)blockIdx.x * blockDim.x + threadIdx.x;
    if (i >= n) return;
    double x = (i % 360) * PI_D / 180.0;
    if      (func == F_SIN)  arr[i] = sin(x);
    else if (func == F_SINF) arr[i] = sinf((float)x);
    else if (func == F_INTR) arr[i] = __sinf((float)x);
}

Appending to lab1.cu


### Part 5. Error on CPU
- Reference: double `sin` for 360 angles (values repeat every 360, no need for 10^9 calls).
- Error = average of |reference − GPU result|.
- The sum is accumulated in double – in float it would stop growing (see the test at the end).

In [ ]:
%%writefile -a lab1.cu

// ===== Part 5. CPU reference and error calculation =====
double REF[360];   // reference: double sin on the CPU

double error_float(const float* h) {
    double sum = 0.0;                       // accumulate in double
    for (long long i = 0; i < N; ++i)
        sum += fabs(REF[i % 360] - (double)h[i]);
    return sum / (double)N;
}

double error_double(const double* h) {
    double sum = 0.0;
    for (long long i = 0; i < N; ++i)
        sum += fabs(REF[i % 360] - h[i]);
    return sum / (double)N;
}

Appending to lab1.cu


### Part 6. Float experiment
- Kernel time: CUDA Events, kernel runs 10 times, time / 10.
- Copy GPU → CPU (`cudaMemcpy` also waits for the kernel to finish), then error on CPU.
- Copy CPU → GPU to measure the other direction.

In [ ]:
%%writefile -a lab1.cu

// ===== Part 6. Experiment with the float array =====
struct Result {
    double err;       // average error
    float kernel_ms;  // kernel time (average over REPEATS launches)
    float d2h_ms;     // copy GPU -> CPU
    float h2d_ms;     // copy CPU -> GPU
};

Result test_float(float* d_arr, float* h_arr, int func) {
    Result r;
    cudaEvent_t start, stop;
    CHECK(cudaEventCreate(&start));
    CHECK(cudaEventCreate(&stop));
    long long blocks = (N + THREADS - 1) / THREADS;
    size_t bytes = N * sizeof(float);

    // 1. Kernel time
    CHECK(cudaEventRecord(start));
    for (int k = 0; k < REPEATS; ++k)
        kernel_float<<<(unsigned)blocks, THREADS>>>(d_arr, N, func);
    CHECK(cudaEventRecord(stop));
    CHECK(cudaEventSynchronize(stop));
    CHECK(cudaGetLastError());
    CHECK(cudaEventElapsedTime(&r.kernel_ms, start, stop));
    r.kernel_ms /= REPEATS;

    // 2. Copy GPU -> CPU
    CHECK(cudaEventRecord(start));
    CHECK(cudaMemcpy(h_arr, d_arr, bytes, cudaMemcpyDeviceToHost));
    CHECK(cudaEventRecord(stop));
    CHECK(cudaEventSynchronize(stop));
    CHECK(cudaEventElapsedTime(&r.d2h_ms, start, stop));

    // 3. Error
    r.err = error_float(h_arr);

    // 4. Copy CPU -> GPU
    CHECK(cudaEventRecord(start));
    CHECK(cudaMemcpy(d_arr, h_arr, bytes, cudaMemcpyHostToDevice));
    CHECK(cudaEventRecord(stop));
    CHECK(cudaEventSynchronize(stop));
    CHECK(cudaEventElapsedTime(&r.h2d_ms, start, stop));

    CHECK(cudaEventDestroy(start));
    CHECK(cudaEventDestroy(stop));
    return r;
}

Appending to lab1.cu


### Part 7. Double experiment
- The same steps for the double array (8 GB, 2 times more data).

In [ ]:
%%writefile -a lab1.cu

// ===== Part 7. Experiment with the double array =====
Result test_double(double* d_arr, double* h_arr, int func) {
    Result r;
    cudaEvent_t start, stop;
    CHECK(cudaEventCreate(&start));
    CHECK(cudaEventCreate(&stop));
    long long blocks = (N + THREADS - 1) / THREADS;
    size_t bytes = N * sizeof(double);

    // 1. Kernel time
    CHECK(cudaEventRecord(start));
    for (int k = 0; k < REPEATS; ++k)
        kernel_double<<<(unsigned)blocks, THREADS>>>(d_arr, N, func);
    CHECK(cudaEventRecord(stop));
    CHECK(cudaEventSynchronize(stop));
    CHECK(cudaGetLastError());
    CHECK(cudaEventElapsedTime(&r.kernel_ms, start, stop));
    r.kernel_ms /= REPEATS;

    // 2. Copy GPU -> CPU
    CHECK(cudaEventRecord(start));
    CHECK(cudaMemcpy(h_arr, d_arr, bytes, cudaMemcpyDeviceToHost));
    CHECK(cudaEventRecord(stop));
    CHECK(cudaEventSynchronize(stop));
    CHECK(cudaEventElapsedTime(&r.d2h_ms, start, stop));

    // 3. Error
    r.err = error_double(h_arr);

    // 4. Copy CPU -> GPU
    CHECK(cudaEventRecord(start));
    CHECK(cudaMemcpy(d_arr, h_arr, bytes, cudaMemcpyHostToDevice));
    CHECK(cudaEventRecord(stop));
    CHECK(cudaEventSynchronize(stop));
    CHECK(cudaEventElapsedTime(&r.h2d_ms, start, stop));

    CHECK(cudaEventDestroy(start));
    CHECK(cudaEventDestroy(stop));
    return r;
}

Appending to lab1.cu


### Part 8. main
- `cudaMalloc` – GPU memory (`d_`), `malloc` – CPU memory (`h_`).
- Warm-up – one kernel launch and one GPU → CPU copy without measuring. The first kernel is slower (CUDA initialization), and the first copy into new CPU memory is ~3 times slower (the OS gives memory pages only at the first write).
- 3 experiments for float, free memory, 3 experiments for double, print tables.

In [ ]:
%%writefile -a lab1.cu

// ===== Part 8. Main function =====
void print_table(const char* title, Result* r, size_t bytes) {
    printf("\n=== %s ===\n", title);
    printf("Function |  err          | kernel, ms | GPU->CPU, ms | CPU->GPU, ms\n");
    printf("---------+---------------+------------+--------------+-------------\n");
    for (int f = 0; f < 3; ++f)
        printf("%s   | %.6e  | %10.3f | %12.1f | %12.1f\n",
               NAMES[f], r[f].err, r[f].kernel_ms, r[f].d2h_ms, r[f].h2d_ms);
    printf("Copy speed GPU->CPU: %.2f GB/s, CPU->GPU: %.2f GB/s\n",
           bytes / (r[0].d2h_ms * 1e6), bytes / (r[0].h2d_ms * 1e6));
}

int main() {
    // 1. CPU reference (double)
    for (int k = 0; k < 360; ++k)
        REF[k] = sin(k * PI_D / 180.0);

    // 2. GPU info
    cudaDeviceProp prop;
    CHECK(cudaGetDeviceProperties(&prop, 0));
    printf("GPU: %s, memory %.1f GB\n", prop.name, prop.totalGlobalMem / 1e9);
    printf("N = %lld\n", N);
    long long blocks = (N + THREADS - 1) / THREADS;

    // 3. float: memory allocation (4 GB on GPU and 4 GB on CPU)
    size_t bytesF = N * sizeof(float);
    float* d_f = nullptr;
    CHECK(cudaMalloc(&d_f, bytesF));
    float* h_f = (float*)malloc(bytesF);
    if (!h_f) { fprintf(stderr, "malloc error (float)\n"); return 1; }

    // 4. Warm-up: one kernel launch and one copy without measuring
    kernel_float<<<(unsigned)blocks, THREADS>>>(d_f, N, F_SINF);
    CHECK(cudaDeviceSynchronize());
    CHECK(cudaMemcpy(h_f, d_f, bytesF, cudaMemcpyDeviceToHost));

    // 5. float experiments
    Result resF[3];
    for (int f = 0; f < 3; ++f)
        resF[f] = test_float(d_f, h_f, f);
    CHECK(cudaFree(d_f));
    free(h_f);

    // 6. double: memory allocation (8 GB on GPU and 8 GB on CPU)
    size_t bytesD = N * sizeof(double);
    double* d_d = nullptr;
    CHECK(cudaMalloc(&d_d, bytesD));
    double* h_d = (double*)malloc(bytesD);
    if (!h_d) { fprintf(stderr, "malloc error (double)\n"); return 1; }

    // warm-up: one kernel launch and one copy without measuring
    kernel_double<<<(unsigned)blocks, THREADS>>>(d_d, N, F_SIN);
    CHECK(cudaDeviceSynchronize());
    CHECK(cudaMemcpy(h_d, d_d, bytesD, cudaMemcpyDeviceToHost));

    Result resD[3];
    for (int f = 0; f < 3; ++f)
        resD[f] = test_double(d_d, h_d, f);
    CHECK(cudaFree(d_d));
    free(h_d);

    // 7. Results
    print_table("float array (4 GB)", resF, bytesF);
    print_table("double array (8 GB)", resD, bytesD);
    return 0;
}

Appending to lab1.cu


### Compile
- `-O2` – optimization, `-arch=sm_75` – Tesla T4.
- Without `-use_fast_math`, otherwise sinf becomes __sinf.

In [ ]:
!nvcc -O2 -arch=sm_75 lab1.cu -o lab1_exec

### Run
- Takes 1–3 minutes (error calculation on CPU for 10^9 elements).

In [ ]:
!./lab1_exec

GPU: Tesla T4, memory 15.6 GB
N = 1000000000

=== float array (4 GB) ===
Function |  err          | kernel, ms | GPU->CPU, ms | CPU->GPU, ms
---------+---------------+------------+--------------+-------------
sin      | 7.599349e-08  |     29.119 |        898.7 |        931.2
sinf     | 7.599349e-08  |     30.583 |        938.0 |       1301.1
__sinf   | 1.148449e-07  |     25.984 |       1146.1 |        881.0
Copy speed GPU->CPU: 4.45 GB/s, CPU->GPU: 4.30 GB/s

=== double array (8 GB) ===
Function |  err          | kernel, ms | GPU->CPU, ms | CPU->GPU, ms
---------+---------------+------------+--------------+-------------
sin      | 8.779628e-18  |    275.462 |       1884.5 |       1727.2
sinf     | 4.679493e-08  |    130.571 |       1720.5 |       1718.3
__sinf   | 1.301495e-07  |    128.296 |       1882.6 |       1738.5
Copy speed GPU->CPU: 4.25 GB/s, CPU->GPU: 4.63 GB/s


### Notes on results
- float: sin = sinf (same code, same error). __sinf – largest error. Kernel times of all 3 functions are close (≈23–31 ms) because the kernel is memory bandwidth bound; small differences between runs are measurement noise.
- double: sin – almost exact (~10^-18) but slowest (few FP64 units on T4, 2× data).
- Copy CPU ↔ GPU takes much longer than the kernel; double copy is ~2 times longer.

### Summation test: float vs double
- Add 0.00001 10^9 times, expected ≈ 10 000.
- Float keeps ~7 digits, so the sum stops at 256. Double keeps 15–17 digits, so the sum is correct.

In [ ]:
%%writefile accum.cpp
#include <iostream>
#include <cmath>
#include <iomanip>

int main() {
    const size_t N = 1000000000;          // 10^9
    const float small_value = 0.00001f;   // small "error" of one element
    const double exact_sum = static_cast<double>(N) * static_cast<double>(small_value);

    std::cout << "Expected sum: " << std::fixed << std::setprecision(4) << exact_sum << std::endl;

    // 1. Accumulate in float
    float float_accumulator = 0.0f;
    for (size_t i = 0; i < N; ++i)
        float_accumulator += small_value;

    // 2. Accumulate in double
    double double_accumulator = 0.0;
    for (size_t i = 0; i < N; ++i)
        double_accumulator += static_cast<double>(small_value);

    std::cout << "Float  accumulator: " << float_accumulator << std::endl;
    std::cout << "Double accumulator: " << double_accumulator << std::endl;
    std::cout << "Float  error: " << std::abs(exact_sum - (double)float_accumulator) << std::endl;
    std::cout << "Double error: " << std::abs(exact_sum - double_accumulator) << std::endl;
    return 0;
}

Writing accum.cpp


In [ ]:
!g++ -O2 accum.cpp -o accum && ./accum

Expected sum: 9999.9997
Float  accumulator: 256.0000
Double accumulator: 9999.9997
Float  error: 9743.9997
Double error: 0.0000
